# Retail Repurchase Prediction: train and deploy on Amazon SageMaker
Predicts whether a customer will buy again in the next 90 days, so marketing can target the customers most likely to return.

**Run cells in order.** Before starting, upload `features_2011_09_01.csv` (from `data/processed/` in the repository) into the same folder as this notebook (JupyterLab upload button).

## 1. Setup

In [ ]:
import boto3, sagemaker, pandas as pd, numpy as np
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()   # or paste the lab bucket name here
prefix = 'retail-repurchase'
print(region, bucket)

## 2. Load data and split

In [ ]:
df = pd.read_csv('features_2011_09_01.csv')
FEATURES = ['recency_days','frequency','monetary','avg_order_value','unique_products',
            'tenure_days','return_rate','orders_last_90d','avg_days_between_orders','is_uk']
# The SageMaker CSV format has no missing values: one-order customers get -1 as the gap
df['avg_days_between_orders'] = df['avg_days_between_orders'].fillna(-1)
print(df.shape, 'repurchase rate:', round(df['label'].mean(), 3))

train, rest = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val, test = train_test_split(rest, test_size=0.5, stratify=rest['label'], random_state=42)

# SageMaker built-in XGBoost: label in first column, no header
train[['label'] + FEATURES].to_csv('train.csv', header=False, index=False)
val[['label'] + FEATURES].to_csv('validation.csv', header=False, index=False)
print(len(train), len(val), len(test))

## 3. Upload to S3

In [ ]:
train_s3 = session.upload_data('train.csv', bucket, f'{prefix}/train')
val_s3 = session.upload_data('validation.csv', bucket, f'{prefix}/validation')
print(train_s3, val_s3)

## 4. Train XGBoost

In [ ]:
image = image_uris.retrieve('xgboost', region, '1.7-1')
xgb = sagemaker.estimator.Estimator(
    image_uri=image, role=role, instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session)
pos_weight = round((train['label'] == 0).sum() / max((train['label'] == 1).sum(), 1), 2)
xgb.set_hyperparameters(objective='binary:logistic', eval_metric='auc',
                        num_round=200, max_depth=4, eta=0.1,
                        subsample=0.8, scale_pos_weight=pos_weight,
                        early_stopping_rounds=20)
xgb.fit({'train': TrainingInput(train_s3, content_type='text/csv'),
         'validation': TrainingInput(val_s3, content_type='text/csv')})

## 5. Deploy the endpoint
Option A (serverless) only bills per request. If the lab blocks it, use Option B.

In [ ]:
# Option A: serverless
from sagemaker.serverless import ServerlessInferenceConfig
predictor = xgb.deploy(
    serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2),
    serializer=CSVSerializer())

# Option B: real-time (comment Option A, uncomment this)
# predictor = xgb.deploy(1, 'ml.m5.large', serializer=CSVSerializer())

print('ENDPOINT_NAME =', predictor.endpoint_name)

## 6. Evaluate on the held-out test set

In [ ]:
raw = predictor.predict(test[FEATURES].values).decode('utf-8')
probs = np.array([float(x) for x in raw.replace('\n', ',').split(',') if x.strip()])
test = test.assign(prob=probs)

auc = roc_auc_score(test['label'], test['prob'])
top = test.nlargest(int(len(test) * 0.2), 'prob')
capture = top['label'].sum() / test['label'].sum()
lift = top['label'].mean() / test['label'].mean()

print(f'AUC: {auc:.3f}')
print(f'Targeting the top 20% of customers reaches {capture:.0%} of all repurchasers')
print(f'Lift over random targeting: {lift:.1f}x')

## 7. Sample request for Lambda / API Gateway

In [ ]:
import json
sample = test.iloc[0]
print(json.dumps({f: float(sample[f]) for f in FEATURES}, indent=2))

## 8. Clean up (only after recording the demo)

In [ ]:
# predictor.delete_endpoint()